# 04 Model Training and Evaluation

Compare models using rare-event metrics and validate investigation-capacity tradeoffs.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Project\Data Science\financial_fraud_detection_&_transaction_intelligence")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.config import REPORT_DIR
from src.visualization import APPROVED_PALETTE

In [3]:
model_summary = json.loads((REPORT_DIR / "model_performance_summary.json").read_text(encoding="utf-8"))
model_summary["best_model"], model_summary["best_model_metrics"]

('Random Forest',
 {'threshold': 0.75,
  'pr_auc': 0.9997917196604756,
  'roc_auc': 0.9999921142999258,
  'precision': 1.0,
  'recall': 0.9993279569892473,
  'f1': 0.9996638655462184,
  'f2': 0.9994622933189945,
  'true_negatives': 1576313,
  'false_positives': 0,
  'false_negatives': 3,
  'true_positives': 4461,
  'false_positive_rate': 0.0})

In [4]:
comparison = pd.read_csv(REPORT_DIR / "model_comparison.csv")
comparison[["model_name", "feature_set", "pr_auc", "roc_auc", "precision", "recall", "f2", "false_positives", "false_negatives"]]

,model_name,feature_set,pr_auc,roc_auc,precision,recall,f2,false_positives,false_negatives
0,Random Forest,final_no_legacy_flag,0.999792,0.999992,1.000000,0.999328,0.999462,0,3
1,HistGradientBoosting + isFlaggedFraud benchmark,benchmark_with_legacy_flag,0.999387,0.999964,1.000000,0.976254,0.980913,0,106
2,HistGradientBoosting,final_no_legacy_flag,0.999387,0.999964,1.000000,0.976254,0.980913,0,106
3,Logistic Regression,final_no_legacy_flag,0.013825,0.898996,0.013825,1.000000,0.065503,318429,0
4,Decision Tree,final_no_legacy_flag,0.002824,0.500000,0.002824,1.000000,0.013962,1576313,0


In [5]:
final_models = comparison[comparison["feature_set"] == "final_no_legacy_flag"].sort_values("pr_auc")
px.bar(
    final_models,
    x="pr_auc",
    y="model_name",
    orientation="h",
    text=final_models["pr_auc"].map(lambda value: f"{value:.4f}"),
    title="Model Comparison by PR-AUC",
    color_discrete_sequence=[APPROVED_PALETTE["accent_blue"]],
)

In [6]:
threshold = pd.read_csv(REPORT_DIR / "threshold_optimization_top50.csv").sort_values("threshold")
fig = go.Figure()
fig.add_scatter(x=threshold["threshold"], y=threshold["precision"], mode="lines", name="Precision")
fig.add_scatter(x=threshold["threshold"], y=threshold["recall"], mode="lines", name="Recall")
fig.add_scatter(x=threshold["threshold"], y=threshold["f2"], mode="lines", name="F2")
fig.update_layout(title="Threshold Tradeoff", xaxis_title="Threshold", yaxis_title="Score")
fig

In [7]:
pd.read_csv(REPORT_DIR / "top_k_fraud_capture.csv")

,review_rate,reviewed_transactions,fraud_captured,precision_at_k,recall_at_k,lift_vs_random
0,0.001,1581,1581,1.000000,0.354167,354.116711
1,0.005,7904,4463,0.564651,0.999776,199.952288
2,0.010,15808,4463,0.282325,0.999776,99.976144
3,0.050,79039,4464,0.056478,1.000000,19.999962
